# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
#your code here
new_spaceship = spaceship.dropna()
new_spaceship.shape


(6606, 14)

In [4]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# Extracts the first letter (deck). If it is does not exist. adds "U" (unknown).
new_spaceship['Deck'] = new_spaceship['Cabin'].astype('string').str.extract(r'([A-Za-z])')
new_spaceship['Deck'] = new_spaceship['Deck'].fillna('U')

# One-hot encode only for this column
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)  # use sparse=True se quiser matriz esparsa
X_cat = ohe.fit_transform(new_spaceship[['Deck']])
cols = ohe.get_feature_names_out(['Deck'])

# Se quiser voltar a um DataFrame:
X_cat_df = pd.DataFrame(X_cat, columns=cols, index=new_spaceship.index)

C:\Users\sofia\AppData\Local\Temp\ipykernel_30576\1565718744.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_spaceship['Deck'] = new_spaceship['Cabin'].astype('string').str.extract(r'([A-Za-z])')
C:\Users\sofia\AppData\Local\Temp\ipykernel_30576\1565718744.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_spaceship['Deck'] = new_spaceship['Deck'].fillna('U')


In [5]:
new_spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,Deck
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False,B
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True,F
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False,A
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False,A
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True,F


In [6]:
# drop PassengerId and Name columns

new_spaceship.drop(columns=['PassengerId', 'Name', 'Cabin'], inplace=True)

C:\Users\sofia\AppData\Local\Temp\ipykernel_30576\2810245407.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_spaceship.drop(columns=['PassengerId', 'Name', 'Cabin'], inplace=True)


In [7]:
df = new_spaceship.copy()

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [8]:
#your code here
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import BaggingClassifier, BaggingRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error, r2_score
import numpy as np
import pandas as pd

In [9]:
#your code here
# perform train_test_split      
from sklearn.model_selection import train_test_split



num_features = ['Age','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
cat_features = ['CryoSleep','Destination','VIP','Deck'] #'HomePlanet'
target = 'Transported'

X = df[num_features + cat_features]
y = df[target].astype(int)  # True/False -> 1/0

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [11]:
# get_dummies for the rest of categorical columns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer


numeric = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    # junta categorias raras para evitar explosão de colunas
    ('ohe', OneHotEncoder(handle_unknown='ignore', min_frequency=0.01))
])

prep = ColumnTransformer([
    ('num', numeric, num_features),
    ('cat', categorical, cat_features)
])

- Evaluate your model

In [12]:
#your code here

cat_cols = X_train.select_dtypes(include=['object','category','string']).columns
num_cols = X_train.columns.difference(cat_cols)

preprocess = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])

In [13]:
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier

hgb_clf = Pipeline([
    ('prep', preprocess),
    ('model', HistGradientBoostingClassifier(
        learning_rate=0.1, max_leaf_nodes=31, random_state=42
    ))
])


hgb_clf.fit(X_train, y_train)
print("GB accuracy:", hgb_clf.score(X_test, y_test))

GB accuracy: 0.8101361573373677


**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [14]:
#your code here

# === HGB: RandomizedSearch + avaliação final ===
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.preprocessing import FunctionTransformer
from scipy import sparse
from scipy.stats import loguniform, randint
import numpy as np


to_dense = FunctionTransformer(lambda X: X.toarray() if sparse.issparse(X) else X)

hgb_clf = Pipeline([
    ('prep', preprocess),   
    ('to_dense', to_dense), 
    ('model', HistGradientBoostingClassifier(
        random_state=42,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20
    ))
])


param_distributions = {
    'model__learning_rate': loguniform(1e-2, 3e-1),   
    'model__max_iter': randint(200, 1001),            
    'model__max_leaf_nodes': randint(15, 128),
    'model__min_samples_leaf': randint(1, 31),
    'model__l2_regularization': loguniform(1e-8, 1e-1),
    'model__max_bins': randint(64, 256)              
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rs = RandomizedSearchCV(
    estimator=hgb_clf,
    param_distributions=param_distributions,
    n_iter=40,                   # sobe para 60–100 se quiseres explorar mais
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

rs.fit(X_train, y_train)

print("Best CV accuracy:", rs.best_score_)
print("Best params:")
for k, v in rs.best_params_.items():
    print(f"  {k}: {v}")

best_hgb = rs.best_estimator_

# --- Avaliação no hold-out ---
y_pred = best_hgb.predict(X_test)
acc = accuracy_score(y_test, y_pred)
cm  = confusion_matrix(y_test, y_pred)
print(f"\nTEST accuracy: {acc:.4f}")
print("Confusion matrix:\n", cm)
print("\nClassification report:\n", classification_report(y_test, y_pred, digits=4))

try:
    y_proba = best_hgb.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    print(f"TEST ROC-AUC: {auc:.4f}")
except Exception:
    pass  # sem stress se não houver predict_proba


Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best CV accuracy: 0.7976930492245061
Best params:
  model__l2_regularization: 1.6351310838425153e-07
  model__learning_rate: 0.03781324384545034
  model__max_bins: 113
  model__max_iter: 587
  model__max_leaf_nodes: 16
  model__min_samples_leaf: 30

TEST accuracy: 0.8116
Confusion matrix:
 [[497 156]
 [ 93 576]]

Classification report:
               precision    recall  f1-score   support

           0     0.8424    0.7611    0.7997       653
           1     0.7869    0.8610    0.8223       669

    accuracy                         0.8116      1322
   macro avg     0.8146    0.8110    0.8110      1322
weighted avg     0.8143    0.8116    0.8111      1322

TEST ROC-AUC: 0.8893


- Run Grid Search

In [16]:
# === HGB: GridSearchCV ===
# 1) REMOVE o to_dense — deixa isto fora
# from sklearn.preprocessing import FunctionTransformer
# from scipy import sparse
# to_dense = FunctionTransformer(lambda X: X.toarray() if sparse.issparse(X) else X)

from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from joblib import Memory
from tempfile import mkdtemp

# IMPORTANTE: no teu 'preprocess', garante:
# OneHotEncoder(handle_unknown='ignore', min_frequency=0.01, sparse_output=False)

# 2) Pipeline com cache para não refazer preprocess a cada combinação
cachedir = mkdtemp()
hgb_clf = Pipeline([
    ('prep', preprocess),  # ColumnTransformer com OHE denso
    # ('to_dense', to_dense),  # <- REMOVIDO
    ('model', HistGradientBoostingClassifier(
        random_state=42,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20
    ))
], memory=Memory(cachedir))

# 3) CV mais leve
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Grid enxuto (32 combinações)
param_grid = {
    'model__learning_rate'    : [0.05, 0.1],
    'model__max_iter'         : [300, 500, 800],
    'model__max_leaf_nodes'   : [31, 63],
    'model__min_samples_leaf' : [1, 5],
    'model__l2_regularization': [0.0, 1e-3],
    # fixa bins para acelerar; mexe nisto depois se precisares
    # 'model__max_bins'       : [128]  
}

gs = GridSearchCV(
    estimator=hgb_clf,
    param_grid=param_grid,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
    verbose=2,
    return_train_score=False
)

gs.fit(X_train, y_train)

print("Best CV accuracy:", gs.best_score_)
print("Best params:")
for k, v in gs.best_params_.items():
    print(f"  {k}: {v}")

best_hgb = gs.best_estimator_

y_pred = best_hgb.predict(X_test)
print(f"\nTEST accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred, digits=4))


Fitting 3 folds for each of 48 candidates, totalling 144 fits
________________________________________________________________________________
[Memory] Calling sklearn.pipeline._fit_transform_one...
_fit_transform_one(ColumnTransformer(transformers=[('num', SimpleImputer(strategy='median'),
                                 Index(['Age', 'FoodCourt', 'RoomService', 'ShoppingMall', 'Spa', 'VRDeck'], dtype='object')),
                                ('cat',
                                 Pipeline(steps=[('imp',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('ohe',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 Index(['CryoSleep', 'Destination', 'VIP', 'Deck'], dtype='object'))]), 
       Age  RoomService  FoodCourt  ShoppingMall     Spa  VRDeck CryoSleep  \
7832  25.0          0.0     1673.0           0.0   642.0   

- Evaluate your model

In [17]:
# === 1) Métricas no hold-out ===
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import numpy as np

y_pred  = best_hgb.predict(X_test)
acc     = accuracy_score(y_test, y_pred)
f1      = f1_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f} | F1: {f1:.4f}")

# Probabilidades (se disponível)
try:
    y_proba = best_hgb.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    print(f"ROC-AUC: {auc:.4f}")
except Exception:
    y_proba = None

print("\nClassification report:\n", classification_report(y_test, y_pred, digits=4))

# Matriz de confusão
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)


Accuracy: 0.8101 | F1: 0.8196
ROC-AUC: 0.8865

Classification report:
               precision    recall  f1-score   support

           0     0.8350    0.7672    0.7997       653
           1     0.7895    0.8520    0.8196       669

    accuracy                         0.8101      1322
   macro avg     0.8122    0.8096    0.8096      1322
weighted avg     0.8120    0.8101    0.8097      1322

Confusion matrix:
 [[501 152]
 [ 99 570]]
